<a href="https://colab.research.google.com/github/david-levin11/Verification_Notebooks/blob/main/petss_twl_forecast_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PETSS Total Water Level Forecast Graphic

This Google Colab notebook downloads a PETSS station CSV tarball from NOMADS, extracts one station, optionally converts water levels from MLLW to MHHW, and creates a customer-facing water level forecast graphic.

The graphic supports configurable time windows, Alaska local time, plain-language labels, and a peak higher-scenario callout.

For problems with this tool or questions about how to use, please email David Levin (david.levin@noaa.gov)


In [ ]:
#@title Configuration
from __future__ import annotations

import difflib
import importlib.util
import json
import re
import tarfile
import textwrap
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from zoneinfo import ZoneInfo
import zipfile
from pathlib import Path
from urllib.request import urlretrieve
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
#@markdown **PETSS run selection.**
DATE = "2026-08-04"  #@param {type: "date"}
CYCLE = "06" #@param ["00", "06", "12", "18"] {type:"string"}

#@markdown **Station selection. Can use station names as seen on the PETSS website or you can also use the site ids as show on the PETSS graphics on the web (ex: 9468756).**
STATION_ID = None    # Example: '9468756'
LOCATION = "Hooper Bay" #@param {type:"string"}
REGION = 'nwak'

# Local time and plot range
TIMEZONE = 'America/Anchorage'

#@markdown **PLOT_START_MODE options:** 'run_time' or 'custom_local'.  Selecting 'run_time' will plot the entire model run.  Selecting 'custom_local' allows you to set your own start and end times for the plots (provided they exist within the model forecast projections)
PLOT_START_MODE = 'custom_local' #@param ["run_time", "custom_local"]
#@markdown If you selected custom_local, enter the start and end times of your graphic in local Alaska time
CUSTOM_START_DATE = "2026-08-04"  #@param {type:"date"}
CUSTOM_START_HOUR = 6             #@param {type:"slider", min:0, max:23, step:1}
CUSTOM_START_MINUTE = 0           #@param {type:"slider", min:0, max:55, step:5}

CUSTOM_END_DATE = "2026-08-05"    #@param {type:"date"}
CUSTOM_END_HOUR = 18              #@param {type:"slider", min:0, max:23, step:1}
CUSTOM_END_MINUTE = 0             #@param {type:"slider", min:0, max:55, step:5}

CUSTOM_START_LOCAL = f"{CUSTOM_START_DATE} {CUSTOM_START_HOUR:02d}:{CUSTOM_START_MINUTE:02d}"
CUSTOM_END_LOCAL = f"{CUSTOM_END_DATE} {CUSTOM_END_HOUR:02d}:{CUSTOM_END_MINUTE:02d}"
# if user selects run_time we plot the whole model run
if PLOT_START_MODE == "run_time":
  CUSTOM_START_LOCAL = None
  CUSTOM_END_LOCAL = None
else:
  CUSTOM_START_LOCAL = CUSTOM_START_LOCAL
  CUSTOM_END_LOCAL = CUSTOM_END_LOCAL

MIN_FORECAST_HOUR = None    # Example: 0
MAX_FORECAST_HOUR = None    # Example: 72

#@markdown **Vertical datum. PETSS is assumed to start as MLLW.**
VERTICAL_DATUM = "MHHW" #@param ["MHHW", "MLLW"]
DATUM_LOOKUP_CSV = 'petss_station_datums.csv'  # or 'sites_and_datums.csv'

# Station map
STATION_MAP_PY = 'petss_station_map.py'

# Output
OUTDIR = 'petss_data'
SAVE_CLEAN_CSV = True
SAVE_PLOT = True

#@markdown **Customer-facing plot labels**
MAIN_LABEL = "Expected water level" #@param {type: "string"}
LOWER_LABEL = "Lower possible outcome" #@param {type: "string"}
HIGHER_LABEL = "Higher possible outcome" #@param {type: "string"}
SHOW_SCENARIO_LINES = True #@param {type: "boolean"}
SHOW_PEAK_CALLOUT = True #@param {type: "boolean"}
CUSTOM_TITLE = "Test Graphic Title" #@param {type:"string"}

if CUSTOM_TITLE == "":
  CUSTOM_TITLE = None
else:
  CUSTOM_TITLE = CUSTOM_TITLE

# Accessing static files on github
STATIC_DIR = Path("petss_static")
STATIC_DIR.mkdir(exist_ok=True)

STATIC_ZIP_URL = "https://github.com/david-levin11/PETSS_Graphics/releases/latest/download/petss_static_files.zip"

STATIC_ZIP = Path("petss_static_files.zip")

urlretrieve(STATIC_ZIP_URL, STATIC_ZIP)

with zipfile.ZipFile(STATIC_ZIP, "r") as z:
    z.extractall(STATIC_DIR)

STATION_MAP_PY = STATIC_DIR / "petss_station_map.py"
DATUM_LOOKUP_CSV = STATIC_DIR / "petss_station_datums.csv"

#print("Station map:", STATION_MAP_PY)
#print("Datum lookup:", DATUM_LOOKUP_CSV)


#print("Station map exists:", Path(STATION_MAP_PY).exists(), STATION_MAP_PY)
#print("Datum lookup exists:", Path(DATUM_LOOKUP_CSV).exists(), DATUM_LOOKUP_CSV)

#for path in Path("petss_static").rglob("*"):
#    print(path)

BASE_URL = 'https://nomads.ncep.noaa.gov/pub/data/nccf/com/petss/prod'
RUN_CYCLES = ('00', '06', '12', '18')
MISSING_VALUES = [9999, 9999.0, 9999.000, -9999, -9999.0]

FALLBACK_STATION_MAP = {
    'nome': '9468756', 'nome norton sound': '9468756', 'prudhoe bay': '9497645',
    'red dog': '9491094', 'red dog dock': '9491094', 'shishmaref': '9469854',
    'unalakleet': '9468333', 'st michael': '9468132', 'kaktovik': '9499176',
    'point hope': '9491873', 'cape krusenstern': '9490571', 'cape espenberg': '9490096',
    'wales': '9469515', 'tin city': '9469439', 'lost river': '9469338',
    'savoonga': '9468258', 'nunam iqua': '9467551', 'hooper bay': '9466931',
    'mekoryuk': '9466217', 'quinhagak': '9465831', 'platinum': '9465396',
    'unalaska': '9462620', 'nikolski': '9462450', 'cold bay': 'ber0022', 'akutan': 'ber0020',
}

####################### Core Helper Functions ##############################

#@title Core helpers

@dataclass(frozen=True)
class PetssCycle:
    date: str
    cycle: str

    @property
    def tar_name(self) -> str:
        return f"petss.t{self.cycle}z.csv.tar.gz"

    @property
    def directory_url(self) -> str:
        return f"{BASE_URL}/petss.{self.date}"

    @property
    def tar_url(self) -> str:
        return f"{self.directory_url}/{self.tar_name}"

    @property
    def run_time_utc(self) -> pd.Timestamp:
        return pd.to_datetime(f"{self.date}{self.cycle}", format="%Y%m%d%H", utc=True)


@dataclass
class StationMaps:
    default_map: dict[str, str]
    ambiguous_map: dict[str, list[str]]
    metadata: dict[str, dict]
    source: str = "fallback"


@dataclass
class DatumConversion:
    station_id: str
    vertical_datum: str = "MLLW"
    mhhw_minus_mllw_ft: float | None = None
    coops_station_id: str | None = None
    coops_station_name: str | None = None
    datum_epoch: str | None = None
    source: str | None = None


def normalize_name(value: str) -> str:
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[, \-]+ak$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"[, \-]+alaska$", "", text, flags=re.IGNORECASE)
    text = text.replace("&", " and ")
    text = re.sub(r"[#'`’]", "", text)
    text = re.sub(r"[/_.:,;()\[\]{}\-]+", " ", text)
    replacements = {
        r"\bst\b": "st", r"\bpt\b": "point", r"\bent\b": "entrance",
        r"\brv\b": "river", r"\br\b": "river", r"\bsd\b": "sound",
        r"\bi\b": "island", r"\bis\b": "island", r"\bn\b": "north",
        r"\bs\b": "south", r"\be\b": "east", r"\bw\b": "west",
    }
    for pattern, repl in replacements.items():
        text = re.sub(pattern, repl, text)
    return re.sub(r"\s+", " ", text).strip()


def clean_station_display_name(station_label: str | None, station_id: str | None = None) -> str:
    if not station_label:
        return station_id or "Selected location"
    name = str(station_label)
    name = re.sub(r"\s*\([A-Za-z0-9]{4,}\)\s*$", "", name)
    if station_id:
        name = name.replace(str(station_id), "")
    name = name.replace("-", " ")
    name = re.sub(r"\s+", " ", name).strip(" ,")
    name = re.sub(r",?\s*AK$", "", name, flags=re.IGNORECASE).strip(" ,")
    return name.title()


def safe_filename_part(value: str) -> str:
    value = clean_station_display_name(value).lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "selected_location"


def local_time_label(tz_name: str) -> str:
    now_local = pd.Timestamp.now(tz=ZoneInfo(tz_name))
    return now_local.strftime("%Z") or tz_name


def format_local_time(dt, *, include_date: bool = True) -> str:
    ts = pd.Timestamp(dt)
    if include_date:
        return f"{ts.strftime('%a %b')} {ts.day}, {ts.strftime('%I').lstrip('0') or '0'} {ts.strftime('%p')}"
    return f"{ts.strftime('%a')} {ts.strftime('%I').lstrip('0') or '0'} {ts.strftime('%p')}"


def url_exists(url: str, timeout: int = 20) -> bool:
    for method in ("HEAD", "GET"):
        try:
            req = Request(url, method=method, headers={"User-Agent": "petss-colab-notebook/1.0"})
            with urlopen(req, timeout=timeout) as resp:
                return 200 <= resp.status < 400
        except HTTPError as exc:
            if exc.code == 405 and method == "HEAD":
                continue
            return False
        except URLError:
            return False
    return False


def find_latest_cycle(max_days_back: int = 3) -> PetssCycle:
    now = datetime.now(timezone.utc)
    candidate_dates = [(now - timedelta(days=i)).strftime("%Y%m%d") for i in range(max_days_back + 1)]
    candidates = [PetssCycle(date=d, cycle=c) for d in candidate_dates for c in RUN_CYCLES]
    candidates.sort(key=lambda c: c.run_time_utc, reverse=True)
    for candidate in candidates:
        if url_exists(candidate.tar_url):
            return candidate
    raise RuntimeError(f"Could not find a PETSS CSV tarball within the last {max_days_back} days.")


def download_file(url: str, output_path: Path, overwrite: bool = False) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists() and output_path.stat().st_size > 0 and not overwrite:
        print(f"Using cached file: {output_path}")
        return output_path
    print(f"Downloading: {url}")
    req = Request(url, headers={"User-Agent": "petss-colab-notebook/1.0"})
    with urlopen(req, timeout=120) as resp, open(output_path, "wb") as f:
        f.write(resp.read())
    return output_path


######################### Station map and datum helpers ######################

#@title Station map and datum lookup helpers

def load_station_maps_from_py(path: Path) -> StationMaps:
    spec = importlib.util.spec_from_file_location("petss_station_map_dynamic", path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import station map file: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    default_map = getattr(module, "DEFAULT_STATION_MAP", {})
    ambiguous_map = getattr(module, "AMBIGUOUS_STATION_MAP", {})
    metadata = getattr(module, "STATION_METADATA", {})
    return StationMaps(
        default_map={normalize_name(k): str(v) for k, v in default_map.items()},
        ambiguous_map={normalize_name(k): [str(item) for item in v] for k, v in ambiguous_map.items()} if isinstance(ambiguous_map, dict) else {},
        metadata=metadata if isinstance(metadata, dict) else {},
        source=str(path),
    )


def load_station_maps(path: str | Path | None) -> StationMaps:
    if path is not None and Path(path).exists() and Path(path).suffix.lower() == ".py":
        return load_station_maps_from_py(Path(path))
    return StationMaps(
        default_map={normalize_name(k): str(v) for k, v in FALLBACK_STATION_MAP.items()},
        ambiguous_map={}, metadata={}, source="built-in fallback"
    )


def station_label_from_metadata(station_id: str, maps: StationMaps) -> str | None:
    item = maps.metadata.get(str(station_id))
    if isinstance(item, dict):
        name = item.get("station_name")
        if isinstance(name, str) and name.strip():
            return name.strip()
    return None


def resolve_station_id(*, station_id: str | None, location: str | None, station_maps: StationMaps) -> tuple[str, str | None]:
    if station_id:
        sid = str(station_id).strip()
        return sid, station_label_from_metadata(sid, station_maps)
    if not location:
        raise ValueError("Provide either STATION_ID or LOCATION.")
    query_norm = normalize_name(location)
    if query_norm in station_maps.ambiguous_map:
        ids = station_maps.ambiguous_map[query_norm]
        raise ValueError(f"Location '{location}' is ambiguous: {', '.join(ids)}. Use STATION_ID.")
    if query_norm in station_maps.default_map:
        sid = station_maps.default_map[query_norm]
        return sid, station_label_from_metadata(sid, station_maps) or location
    contains = [(alias, sid) for alias, sid in station_maps.default_map.items() if query_norm in alias]
    unique_ids = sorted({sid for _alias, sid in contains})
    if len(unique_ids) == 1:
        sid = unique_ids[0]
        return sid, station_label_from_metadata(sid, station_maps) or location
    if len(unique_ids) > 1:
        choices = ", ".join(f"{alias} ({sid})" for alias, sid in contains[:15])
        raise ValueError(f"Location '{location}' matched multiple stations: {choices}. Use STATION_ID.")
    close = difflib.get_close_matches(query_norm, list(station_maps.default_map.keys()), n=8, cutoff=0.65)
    close_ids = sorted({station_maps.default_map[name] for name in close})
    if len(close_ids) == 1 and close:
        sid = close_ids[0]
        print(f"Matched location '{location}' to alias '{close[0]}' ({sid}) from {station_maps.source}.")
        return sid, station_label_from_metadata(sid, station_maps) or close[0]
    raise ValueError(f"Could not match location '{location}' to a station ID. Use STATION_ID or upload station map.")


def load_datum_lookup(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Datum lookup file not found: {path}. Upload it or set VERTICAL_DATUM='MLLW'.")
    raw = pd.read_csv(path, dtype=str)
    original_columns = list(raw.columns)
    df = raw.copy()
    df.columns = [c.strip().lower().replace(" ", "_").replace("?", "").replace("/", "_").replace("-", "_") for c in df.columns]
    if {"station_id", "mhhw_minus_mllw_ft"}.issubset(df.columns):
        if "status" in df.columns:
            df = df[df["status"].astype(str).str.lower() == "ok"].copy()
        df["station_id"] = df["station_id"].astype(str).str.strip()
        df["mhhw_minus_mllw_ft"] = pd.to_numeric(df["mhhw_minus_mllw_ft"], errors="coerce")
        if "coops_station_id" not in df.columns:
            df["coops_station_id"] = df["station_id"]
        if "coops_station_name" not in df.columns and "station_name" in df.columns:
            df["coops_station_name"] = df["station_name"]
        if "datum_epoch" not in df.columns:
            df["datum_epoch"] = pd.NA
    elif {"station_id", "mhhw", "mllw"}.issubset(df.columns):
        df["station_id"] = df["station_id"].astype(str).str.strip()
        df["mhhw_ft"] = pd.to_numeric(df["mhhw"], errors="coerce")
        df["mllw_ft"] = pd.to_numeric(df["mllw"], errors="coerce")
        df["mhhw_minus_mllw_ft"] = df["mhhw_ft"] - df["mllw_ft"]
        df["coops_station_id"] = df["station_id"]
        df["coops_station_name"] = df["station_name"] if "station_name" in df.columns else pd.NA
        df["datum_epoch"] = df["datum_epoch"] if "datum_epoch" in df.columns else pd.NA
    else:
        raise ValueError("Datum lookup format not recognized. Expected station_id/mhhw_minus_mllw_ft or Station ID/MHHW/MLLW. Found columns: " + ", ".join(original_columns))
    df = df.dropna(subset=["station_id", "mhhw_minus_mllw_ft"])
    df = df.drop_duplicates(subset=["station_id"], keep="first")
    if df.empty:
        raise ValueError(f"No usable datum rows found in {path}")
    return df.set_index("station_id")


def get_datum_conversion(*, station_id: str, vertical_datum: str, datum_lookup_path: str | Path | None) -> DatumConversion:
    requested = vertical_datum.upper()
    if requested == "MLLW":
        return DatumConversion(station_id=station_id, vertical_datum="MLLW", source="no conversion")
    if requested != "MHHW":
        raise ValueError("Only MLLW and MHHW are supported.")
    if datum_lookup_path is None:
        raise ValueError("MHHW conversion requires DATUM_LOOKUP_CSV.")
    lookup = load_datum_lookup(datum_lookup_path)
    sid = str(station_id).strip()
    if sid not in lookup.index:
        raise ValueError(f"No MHHW/MLLW datum lookup row found for station {sid} in {datum_lookup_path}.")
    row = lookup.loc[sid]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    offset = pd.to_numeric(row["mhhw_minus_mllw_ft"], errors="coerce")
    if pd.isna(offset):
        raise ValueError(f"Datum row for station {sid} does not have valid mhhw_minus_mllw_ft.")
    return DatumConversion(
        station_id=sid, vertical_datum="MHHW", mhhw_minus_mllw_ft=float(offset),
        coops_station_id=str(row.get("coops_station_id", "")) or None,
        coops_station_name=str(row.get("coops_station_name", "")) or None,
        datum_epoch=str(row.get("datum_epoch", "")) or None,
        source=str(datum_lookup_path),
    )


def apply_vertical_datum_conversion(df: pd.DataFrame, conversion: DatumConversion) -> pd.DataFrame:
    out = df.copy()
    out["vertical_datum"] = conversion.vertical_datum
    out["mhhw_minus_mllw_ft"] = conversion.mhhw_minus_mllw_ft
    out["datum_source"] = conversion.source
    out["coops_station_id"] = conversion.coops_station_id
    out["coops_station_name"] = conversion.coops_station_name
    out["datum_epoch"] = conversion.datum_epoch
    if conversion.vertical_datum == "MHHW":
        for col in ("TWL", "TWL10p", "TWL90p"):
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce") - conversion.mhhw_minus_mllw_ft
    return out


####################  Download, Read and Parse PETSS Data ####################

#@title Download, extract, and read PETSS station CSV

def safe_tar_member_name(member: tarfile.TarInfo) -> str:
    name = member.name.replace("\\", "/")
    parts = [p for p in name.split("/") if p not in ("", ".")]
    if any(p == ".." for p in parts) or name.startswith("/"):
        raise ValueError(f"Unsafe path in tar archive: {member.name}")
    return "/".join(parts)


def find_station_member(tar_path: Path, station_id: str) -> tarfile.TarInfo:
    target = f"{station_id}.csv".lower()
    with tarfile.open(tar_path, "r:gz") as tf:
        matches = []
        for member in tf.getmembers():
            if not member.isfile():
                continue
            name = safe_tar_member_name(member)
            if Path(name).name.lower() == target:
                matches.append(member)
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(f"Found multiple CSVs for station {station_id}: {[m.name for m in matches]}")
    raise FileNotFoundError(f"Station CSV {station_id}.csv was not found in {tar_path}")


def extract_station_csv(tar_path: Path, station_id: str, output_dir: Path, overwrite: bool = False) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{station_id}.csv"
    if output_path.exists() and output_path.stat().st_size > 0 and not overwrite:
        print(f"Using cached station CSV: {output_path}")
        return output_path
    with tarfile.open(tar_path, "r:gz") as tf:
        member = find_station_member(tar_path, station_id)
        extracted = tf.extractfile(member)
        if extracted is None:
            raise RuntimeError(f"Could not extract {member.name}")
        output_path.write_bytes(extracted.read())
    print(f"Extracted station CSV: {output_path}")
    return output_path


def read_petss_station_csv(csv_path: Path, cycle: PetssCycle, station_id: str, station_name: str | None) -> pd.DataFrame:
    df = pd.read_csv(csv_path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    if "TIME" not in df.columns:
        raise ValueError(f"Expected TIME column in {csv_path}; columns are {list(df.columns)}")
    df["valid_time"] = pd.to_datetime(df["TIME"].astype(str), format="%Y%m%d%H%M", utc=True, errors="coerce")
    numeric_cols = [c for c in df.columns if c not in {"TIME", "valid_time"}]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].where(~df[col].isin(MISSING_VALUES), pd.NA)
    df["run_time"] = cycle.run_time_utc
    df["forecast_hour"] = (df["valid_time"] - cycle.run_time_utc) / pd.Timedelta(hours=1)
    df["station_id"] = str(station_id)
    df["station_name"] = station_name
    first_cols = ["station_id", "station_name", "run_time", "valid_time", "forecast_hour"]
    other_cols = [c for c in df.columns if c not in first_cols]
    return df[first_cols + other_cols]

def normalize_petss_date(value) -> str:
    """
    Accept either Colab date-picker format YYYY-MM-DD
    or PETSS/NOMADS format YYYYMMDD.
    """
    if value is None:
        return None

    text = str(value).strip()

    if re.fullmatch(r"\d{8}", text):
        return text

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", text):
        return pd.Timestamp(text).strftime("%Y%m%d")

    raise ValueError(
        f"DATE must be YYYYMMDD or YYYY-MM-DD, got: {value}"
    )

def get_petss_dataframe() -> tuple[pd.DataFrame, PetssCycle, str, str | None]:
    if DATE and CYCLE:
        cycle = PetssCycle(date=normalize_petss_date(DATE), cycle=f"{int(CYCLE):02d}")
    elif DATE or CYCLE:
        raise ValueError("Set both DATE and CYCLE, or leave both as None.")
    else:
        cycle = find_latest_cycle()
        print(f"Latest available PETSS CSV cycle appears to be {cycle.date} t{cycle.cycle}z")
    station_maps = load_station_maps(STATION_MAP_PY)
    print(f"Station-map source: {station_maps.source}")
    resolved_station_id, resolved_station_name = resolve_station_id(station_id=STATION_ID, location=LOCATION, station_maps=station_maps)
    print(f"Using station: {resolved_station_id}" + (f" ({resolved_station_name})" if resolved_station_name else ""))
    outdir = Path(OUTDIR)
    tar_path = outdir / "raw" / cycle.date / f"t{cycle.cycle}z" / cycle.tar_name
    download_file(cycle.tar_url, tar_path, overwrite=False)
    station_csv_dir = outdir / "stations" / cycle.date / f"t{cycle.cycle}z"
    csv_path = extract_station_csv(tar_path, resolved_station_id, station_csv_dir, overwrite=False)
    df = read_petss_station_csv(csv_path, cycle=cycle, station_id=resolved_station_id, station_name=resolved_station_name)
    datum_lookup = DATUM_LOOKUP_CSV if VERTICAL_DATUM.upper() == "MHHW" else None
    conversion = get_datum_conversion(station_id=resolved_station_id, vertical_datum=VERTICAL_DATUM, datum_lookup_path=datum_lookup)
    df = apply_vertical_datum_conversion(df, conversion)
    if conversion.vertical_datum == "MHHW":
        print(f"Converted TWL/TWL10p/TWL90p from MLLW to MHHW using MHHW-MLLW = {conversion.mhhw_minus_mllw_ft:.2f} ft from {conversion.source}")
    else:
        print("Using PETSS water levels as MLLW-referenced values.")
    return df, cycle, resolved_station_id, resolved_station_name


df, cycle, station_id, station_name = get_petss_dataframe()
#print("\nData preview:")
#display(df.head())

############################  Plotting Helper Functions ######################

#@title Plotting helpers with configurable time range and smart peak callout

def filter_plot_window(df: pd.DataFrame, *, timezone: str, start_mode: str, custom_start_local: str | None, custom_end_local: str | None, min_forecast_hour: float | None, max_forecast_hour: float | None) -> pd.DataFrame:
    tz = ZoneInfo(timezone)
    plot_df = df.sort_values("valid_time").copy()
    plot_df["valid_time"] = pd.to_datetime(plot_df["valid_time"], utc=True, errors="coerce")
    run_time_utc = pd.to_datetime(plot_df["run_time"].dropna().iloc[0], utc=True) if "run_time" in plot_df.columns and plot_df["run_time"].notna().any() else pd.NaT
    if start_mode == "run_time" and pd.notna(run_time_utc):
        plot_df = plot_df[plot_df["valid_time"] >= run_time_utc].copy()
    elif start_mode == "first_valid":
        first_valid = plot_df.dropna(subset=["TWL"])["valid_time"].min()
        plot_df = plot_df[plot_df["valid_time"] >= first_valid].copy()
    elif start_mode == "custom_local":
        if not custom_start_local:
            raise ValueError("PLOT_START_MODE='custom_local' requires CUSTOM_START_LOCAL.")
        start_local = pd.Timestamp(custom_start_local, tz=tz)
        plot_df = plot_df[plot_df["valid_time"] >= start_local.tz_convert("UTC")].copy()
    else:
        raise ValueError("PLOT_START_MODE must be 'run_time', 'first_valid', or 'custom_local'.")
    if custom_end_local:
        end_local = pd.Timestamp(custom_end_local, tz=tz)
        plot_df = plot_df[plot_df["valid_time"] <= end_local.tz_convert("UTC")].copy()
    if min_forecast_hour is not None and "forecast_hour" in plot_df.columns:
        plot_df = plot_df[pd.to_numeric(plot_df["forecast_hour"], errors="coerce") >= min_forecast_hour].copy()
    if max_forecast_hour is not None and "forecast_hour" in plot_df.columns:
        plot_df = plot_df[pd.to_numeric(plot_df["forecast_hour"], errors="coerce") <= max_forecast_hour].copy()
    plot_df["plot_time"] = plot_df["valid_time"].dt.tz_convert(tz)
    return plot_df


def choose_callout_position(peak_time, peak_value, plot_df, ax):
    x_min = plot_df["plot_time"].min()
    x_max = plot_df["plot_time"].max()
    y_min, y_max = ax.get_ylim()
    x_frac = (peak_time - x_min) / (x_max - x_min) if x_max != x_min else 0.5
    y_frac = (peak_value - y_min) / (y_max - y_min) if y_max != y_min else 0.5
    if y_frac > 0.65:
        candidates = [(-85, -35, "right", "top"), (85, -35, "left", "top"), (-85, -55, "right", "top"), (85, -55, "left", "top")]
    else:
        candidates = [(70, 35, "left", "bottom"), (-70, 35, "right", "bottom"), (70, -35, "left", "top"), (-70, -35, "right", "top")]
    if x_frac < 0.35 and y_frac > 0.55:
        candidates = [(90, -45, "left", "top"), (90, -65, "left", "top")] + candidates
    if x_frac > 0.70:
        candidates = [c for c in candidates if c[0] < 0] + [c for c in candidates if c[0] > 0]
    elif x_frac < 0.30:
        candidates = [c for c in candidates if c[0] > 0] + [c for c in candidates if c[0] < 0]
    return candidates[0]


def plot_twl_customer(df: pd.DataFrame, output_path: Path | None = None) -> Path | None:
    tz = ZoneInfo(TIMEZONE)
    tz_label = local_time_label(TIMEZONE)
    plot_df = filter_plot_window(df, timezone=TIMEZONE, start_mode=PLOT_START_MODE, custom_start_local=CUSTOM_START_LOCAL, custom_end_local=CUSTOM_END_LOCAL, min_forecast_hour=MIN_FORECAST_HOUR, max_forecast_hour=MAX_FORECAST_HOUR)
    if plot_df.empty:
        raise ValueError("No data left after applying the configured plot time range.")
    twl_df = plot_df.dropna(subset=["plot_time", "TWL"]).copy()
    fig, ax = plt.subplots(figsize=(11, 7.2))
    peak_time = None
    peak_value = None
    if {"TWL10p", "TWL90p"}.issubset(plot_df.columns):
        pct_df = plot_df.dropna(subset=["plot_time", "TWL10p", "TWL90p"]).copy()
        if not pct_df.empty:
            p10 = pd.to_numeric(pct_df["TWL10p"], errors="coerce")
            p90 = pd.to_numeric(pct_df["TWL90p"], errors="coerce")
            lower = pd.concat([p10, p90], axis=1).min(axis=1)
            upper = pd.concat([p10, p90], axis=1).max(axis=1)
            pct_df = pct_df.assign(_lower=lower, _upper=upper)
            ax.fill_between(pct_df["plot_time"], pct_df["_lower"], pct_df["_upper"], alpha=0.22, label="Possible range", zorder=1)
            if SHOW_SCENARIO_LINES:
                ax.plot(pct_df["plot_time"], pct_df["_lower"], linewidth=1.5, linestyle="--", label=LOWER_LABEL, zorder=2)
                ax.plot(pct_df["plot_time"], pct_df["_upper"], linewidth=1.8, linestyle="--", label=HIGHER_LABEL, zorder=2)
            peak_idx = pct_df["_upper"].idxmax()
            peak_time = pct_df.loc[peak_idx, "plot_time"]
            peak_value = float(pct_df.loc[peak_idx, "_upper"])
    if not twl_df.empty:
        ax.plot(twl_df["plot_time"], pd.to_numeric(twl_df["TWL"], errors="coerce"), linewidth=3, marker="o", markersize=2.5, label=MAIN_LABEL, zorder=3)
        if peak_time is None:
            peak_idx = pd.to_numeric(twl_df["TWL"], errors="coerce").idxmax()
            peak_time = twl_df.loc[peak_idx, "plot_time"]
            peak_value = float(twl_df.loc[peak_idx, "TWL"])
    display_name = clean_station_display_name(station_name or LOCATION or station_id, station_id=station_id)
    vertical_datum = str(plot_df["vertical_datum"].dropna().iloc[0]).upper() if "vertical_datum" in plot_df.columns and plot_df["vertical_datum"].notna().any() else "MLLW"
    if CUSTOM_TITLE:
        title = CUSTOM_TITLE
    else:
        run_time_utc = pd.to_datetime(plot_df["run_time"].dropna().iloc[0], utc=True) if "run_time" in plot_df.columns and plot_df["run_time"].notna().any() else pd.NaT
        if pd.notna(run_time_utc):
            run_time_local = pd.Timestamp(run_time_utc).tz_convert(tz)
            run_str = format_local_time(run_time_local, include_date=True)
            title = f"Water Level Forecast for {display_name}\nForecast issued: {run_str} {tz_label}"
        else:
            title = f"Water Level Forecast for {display_name}"
    ax.set_title(title, fontsize=19, fontweight="bold", pad=14)
    ax.set_xlabel(f"Date and time ({tz_label})", fontsize=12)
    if vertical_datum == "MHHW":
        ax.set_ylabel("Water height above normal high tide (feet)", fontsize=12)
    else:
        ax.set_ylabel(f"Water level above {vertical_datum} (feet)", fontsize=12)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=5, maxticks=9))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H", tz=tz))
    ax.grid(True, alpha=0.35)
    ax.legend(loc="upper left", frameon=True, fontsize=10)
    ax.tick_params(axis="both", labelsize=10)
    ax.margins(x=0.02)
    if SHOW_PEAK_CALLOUT and peak_time is not None and peak_value is not None:
        peak_label_time = format_local_time(peak_time, include_date=False)
        callout = f"Peak higher\nscenario\n{peak_value:.1f} ft above\nnormal high tide\n{peak_label_time} {tz_label}"
        ax.scatter([peak_time], [peak_value], s=55, zorder=5)
        dx, dy, ha, va = choose_callout_position(peak_time, peak_value, plot_df, ax)
        ax.annotate(callout, xy=(peak_time, peak_value), xycoords="data", xytext=(dx, dy), textcoords="offset points", fontsize=9, fontweight="bold", ha=ha, va=va, arrowprops={"arrowstyle": "->", "lw": 1.2, "shrinkA": 4, "shrinkB": 4}, bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "alpha": 0.94, "edgecolor": "0.35"}, annotation_clip=True, zorder=6)
    note = "Shaded area shows a reasonable range of outcomes. Actual water levels may fall outside this range."
    if vertical_datum == "MHHW":
        note += " Reference: normal high tide (MHHW)."
    note = textwrap.fill(note, width=70)
    ax.text(0.01, 0.02, note, transform=ax.transAxes, fontsize=9, va="bottom", ha="left", bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "alpha": 0.82, "edgecolor": "0.8"}, zorder=6)
    fig.autofmt_xdate()
    fig.subplots_adjust(left=0.11, right=0.98, top=0.88, bottom=0.18)
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"Wrote plot: {output_path}")
    plt.show()
    plt.close(fig)
    return output_path

################### Plotting the Data & Saving Output ######################

#@title Create the graphic and save outputs

outdir = Path(OUTDIR)
products_dir = outdir / "products" / cycle.date / f"t{cycle.cycle}z"
products_dir.mkdir(parents=True, exist_ok=True)

datum_label = str(df["vertical_datum"].iloc[0]).lower() if "vertical_datum" in df.columns else "mllw"
display_name = clean_station_display_name(station_name or LOCATION or station_id, station_id=station_id)
location_part = safe_filename_part(display_name)

cleaned_path = products_dir / f"petss_{cycle.date}_t{cycle.cycle}z_{station_id}_{datum_label}_clean.csv"
plot_path = products_dir / f"petss_{cycle.date}_t{cycle.cycle}z_{location_part}_{datum_label}_twl.png"

if SAVE_CLEAN_CSV:
    df.to_csv(cleaned_path, index=False)
    print(f"Wrote cleaned CSV: {cleaned_path}")

if SAVE_PLOT:
    plot_twl_customer(df, plot_path)
else:
    plot_twl_customer(df, None)

print("\nOutput files:")
if SAVE_CLEAN_CSV:
    print(cleaned_path)
if SAVE_PLOT:
    print(plot_path)